In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.append(str(PROJECT_ROOT / "03_codigo" / "utils"))

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== CONFIG GENERAL =====
DATASET = "BCCC17"

ESTADO_ENTRADA = "split"
VERSION_ENTRADA = "v1"

ESTADO_SALIDA = "RUS_SMOTE_ENN"
VERSION_SALIDA = "v1"

LABEL_COL = "LABEL"

TARGET_N = 10000
RANDOM_STATE = 42

# ===== NOMBRES DE SPLITS =====
SPLIT_TRAIN = "train"
SPLIT_TEST = "test"

In [3]:
def construir_nombre_dataset(dataset, estado, version):
    return f"{dataset}__{estado}__{version}"

def construir_nombre_csv(nombre_dataset, split):
    return f"{nombre_dataset}__{split}.csv"

def construir_ruta_base_processed(project_root, nombre_dataset):
    return project_root / "02_datasets" / "processed" / nombre_dataset

def construir_ruta_base_modelos_simples(project_root, nombre_dataset):
    return project_root / "02_datasets" / "processed" / nombre_dataset

In [4]:
nombre_dataset_entrada = construir_nombre_dataset(DATASET, ESTADO_ENTRADA, VERSION_ENTRADA)
nombre_dataset_salida = construir_nombre_dataset(DATASET, ESTADO_SALIDA, VERSION_SALIDA)

nombre_train = construir_nombre_csv(nombre_dataset_entrada, SPLIT_TRAIN)
nombre_test = construir_nombre_csv(nombre_dataset_entrada, SPLIT_TEST)

nombre_train_generado = construir_nombre_csv(nombre_dataset_salida, SPLIT_TRAIN)
nombre_test_generado = construir_nombre_csv(nombre_dataset_salida, SPLIT_TEST)
nombre_test_generado_extra = nombre_test_generado.replace(".csv", "__nuevo.csv")

ruta_base_dataset = construir_ruta_base_processed(PROJECT_ROOT, nombre_dataset_entrada)
ruta_base_dataset_generado = construir_ruta_base_modelos_simples(PROJECT_ROOT, nombre_dataset_salida)

print("=== ENTRADA ===")
print("Dataset:", nombre_dataset_entrada)
print("Train  :", nombre_train)
print("Test   :", nombre_test)
print("Ruta   :", ruta_base_dataset)
print()

print("=== SALIDA ===")
print("Dataset:", nombre_dataset_salida)
print("Train  :", nombre_train_generado)
print("Test   :", nombre_test_generado)
print("Test extra:", nombre_test_generado_extra)
print("Ruta   :", ruta_base_dataset_generado)

=== ENTRADA ===
Dataset: BCCC17__split__v1
Train  : BCCC17__split__v1__train.csv
Test   : BCCC17__split__v1__test.csv
Ruta   : /home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__split__v1

=== SALIDA ===
Dataset: BCCC17__RUS_SMOTE_ENN__v1
Train  : BCCC17__RUS_SMOTE_ENN__v1__train.csv
Test   : BCCC17__RUS_SMOTE_ENN__v1__test.csv
Test extra: BCCC17__RUS_SMOTE_ENN__v1__test__nuevo.csv
Ruta   : /home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1


In [5]:
df_train = cargar_dataset(nombre_train, ruta_base_dataset)
df_test = cargar_dataset(nombre_test, ruta_base_dataset)

In [6]:
print("Distribución inicial TRAIN:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución inicial TEST:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Distribución inicial TRAIN:


,count
LABEL,
0,1372038
1,276914
2,129058
3,76583
4,7625
5,6691
6,5485
7,4759
8,4406


Distribución inicial TEST:


,count
LABEL,
0,343010
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [7]:
df_train_balanceado, df_test_nuevo = balancear_y_mover_a_test(
    df_train, 
    df_test, 
    LABEL_COL, 
    TARGET_N,
    random_state=RANDOM_STATE,
)

df_train_balanceado = aplicar_enn(df_train_balanceado)

In [8]:
print("Distribución final TRAIN balanceado:")
display(df_train_balanceado[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución final TEST original:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución final TEST nuevo:")
display(df_test_nuevo[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Distribución final TRAIN balanceado:


,count
LABEL,
0,10000
8,9991
4,9978
13,9974
2,9973
7,9947
12,9811
9,9620
6,9571


Distribución final TEST original:


,count
LABEL,
0,343010
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


Distribución final TEST nuevo:


,count
LABEL,
0,1705048
1,336143
2,151323
3,85729
4,1906
5,1673
6,1371
7,1190
8,1102


In [9]:
Path(ruta_base_dataset_generado).mkdir(parents=True, exist_ok=True)

print("Ruta de salida creada/verificada:")
print(Path(ruta_base_dataset_generado).resolve())

Ruta de salida creada/verificada:
/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1


In [10]:
guardar_dataset_csv(df_train_balanceado, nombre_train_generado, ruta_base_dataset_generado)
guardar_dataset_csv(df_test, nombre_test_generado, ruta_base_dataset_generado)
guardar_dataset_csv(df_test_nuevo, nombre_test_generado_extra, ruta_base_dataset_generado)

print("Datasets guardados correctamente.")
print()
print("Train balanceado :", Path(ruta_base_dataset_generado) / nombre_train_generado)
print("Test original    :", Path(ruta_base_dataset_generado) / nombre_test_generado)
print("Test nuevo       :", Path(ruta_base_dataset_generado) / nombre_test_generado_extra)

Datasets guardados correctamente.

Train balanceado : /home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__train.csv
Test original    : /home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__test.csv
Test nuevo       : /home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__test__nuevo.csv


In [11]:
print("========== RESUMEN ==========")
print("Dataset entrada :", nombre_dataset_entrada)
print("Dataset salida  :", nombre_dataset_salida)
print()
print("Train original  :", df_train.shape)
print("Test original   :", df_test.shape)
print("Train balanceado:", df_train_balanceado.shape)
print("Test nuevo      :", df_test_nuevo.shape)
print()
print("Parámetros usados:")
print("LABEL_COL       =", LABEL_COL)
print("TARGET_N        =", TARGET_N)
print("RANDOM_STATE    =", RANDOM_STATE)

========== RESUMEN ==========
Dataset entrada : BCCC17__split__v1
Dataset salida  : BCCC17__RUS_SMOTE_ENN__v1

Train original  : (1890959, 64)
Test original   : (472740, 64)
Train balanceado: (131810, 64)
Test nuevo      : (2287333, 64)

Parámetros usados:
LABEL_COL       = LABEL
TARGET_N        = 10000
RANDOM_STATE    = 42
